# Phase 1 - Moltbook A2A Prompt Injection: Data Exploration and Candidate Generation

This notebook is **Phase 1** of the study *"Agent-to-Agent Prompt Injection in AI-Only Social Networks: A Taxonomy and Detection Study on Moltbook"*. Operating on the complete Moltbook Observatory Archive (3,105,136 posts, 27 January - 28 May 2026), it performs the following:

1. Loads the data (download from Hugging Face **or** a previously scanned cache).
2. Generates candidate attack posts through a **high-recall heuristic regex scan** over five categories (C1-C5).
3. Carries out exploratory data analysis (daily volume, submolts, content length) and phase-based attack analysis.
4. Exports a **630-post labelling seed set** via phase-balanced positive sampling plus negative sampling.

**Output:** `moltbook_seed_v2.csv` - manually labelled in Phase 2 and used for model training.

**Appendix A** (at the end): a random-sample audit that measures the recall of the scan.

> **Note:** Run the cells in order. In Section 3, either **Option A** (cache, fast) or **Option B** (download from scratch) is sufficient - do not run both.

## 1. Setup: Google Drive and Folder Structure

Mounts Google Drive and defines the data/output folder paths. Edit `DATASET_DIR` to match your own folder.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# === EDIT THIS TO MATCH YOUR OWN FOLDER ===
# Example: '/content/drive/MyDrive/Moltbook_Research'
PROJECT_DIR = '/content/drive/MyDrive/Moltbook_Research'
# ==========================================

DATASET_DIR = os.path.join(PROJECT_DIR, 'dataset')
FIG_DIR     = os.path.join(PROJECT_DIR, 'fig')
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'output')

for d in [DATASET_DIR, FIG_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'Ready: {d}')

assert os.path.isdir(PROJECT_DIR), f'Project folder not found: {PROJECT_DIR}'
print('\nDrive connection and folder structure ready.')

## 2. Environment Setup

Installs and imports the required libraries (`datasets`, `pandas`, `matplotlib`, `seaborn`, `tqdm`).

In [ ]:
!pip install -q datasets pyarrow pandas numpy matplotlib seaborn tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from datasets import load_dataset
from tqdm.auto import tqdm

pd.set_option('display.max_colwidth', 200)
sns.set_theme(style='whitegrid')

def save_fig(name, dpi=150):
    """Saves the active matplotlib figure to FIG_DIR as a PNG."""
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'Figure saved: {path}')

print('Environment ready.')

## 3. Data Loading

The full archive (3,105,136 posts) can be loaded in two ways - run **only one** of them:
- **Option A** - reads the scanned cache previously saved in Section 4a (takes seconds).
- **Option B** - downloads from Hugging Face from scratch (if no cache exists).

### 3a. Option A - Load from cache (fast)

If `full_archive_scan_cache.parquet` exists, the scanned `df_posts` is loaded directly from it. **If this works, skip Option B and the Section 4 scan and go straight to Section 5.**

In [ ]:
import pandas as pd
import os
import ast

SCAN_CACHE = os.path.join(DATASET_DIR, 'full_archive_scan_cache.parquet')

if os.path.exists(SCAN_CACHE):
    print('Cache found, loading from disk (fast)...')
    df_posts = pd.read_parquet(SCAN_CACHE)
    # Convert attack_hits back from string to dict
    df_posts['attack_hits'] = df_posts['attack_hits_str'].apply(ast.literal_eval)
    df_posts = df_posts.drop(columns=['attack_hits_str'])
    print(f'Loaded: {len(df_posts):,} posts (scan included)')
else:
    print('Cache not found; you need to download and scan from scratch.')

### 3b. Option B - Download from scratch (if no cache exists)

Downloads the Moltbook Observatory Archive from Hugging Face. Afterwards, run the Section 4 scan and the Section 4a cache save.

In [ ]:
"""# Load the 28 January - 9 February slice of posts (the busiest period)
posts_ds = load_dataset(
    'SimulaMet/moltbook-observatory-archive',
    'posts',
    data_files=['data/posts/2026-01-2*.parquet', 'data/posts/2026-02-0*.parquet'],
    split='train'
)"""

posts_ds = load_dataset(
    'SimulaMet/moltbook-observatory-archive',
    'posts',
    split='archive'
)


df_posts = posts_ds.to_pandas()
print(f'Number of posts loaded: {len(df_posts):,}')
print(f'Memory usage: {df_posts.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df_posts.head(3)

### 3c. Data structure and date preparation

Inspects the columns/missing values and derives `date` from `created_at` (for phase assignment and daily analysis). Run this under either option.

In [ ]:
print('--- Columns and dtypes ---')
print(df_posts.dtypes)
print('\n--- Missing value counts ---')
print(df_posts.isna().sum())

In [ ]:
df_posts['created_at'] = pd.to_datetime(df_posts['created_at'], utc=True)
df_posts['date'] = df_posts['created_at'].dt.date
print("Date range:", df_posts['date'].min(), "-->", df_posts['date'].max())
print("Total days:", df_posts['date'].nunique())
print("\nDaily post distribution:")
print(df_posts.groupby('date').size().to_string())

## 4. Heuristic Scan: Candidate Attack Generation

Since manually labelling 3.1M posts is not feasible, a **high-recall** regex scan extracts a pool of candidate attacks. Surface-level cue patterns are defined for the five taxonomy categories (C1-C5); every post is scanned against all patterns, and posts with at least one match are treated as **candidates**. This cell defines the regex patterns and the `scan()` function; the next cell adds the phase label, runs the scan and produces `df_candidates`. *(The recall of this stage is audited in Appendix A.)*

In [ ]:
import re

attack_patterns = {
    'C1_instruction_override': [
        r'ignore (all |your |the )?(previous|prior|above) (instructions?|prompts?)',
        r'disregard (your|the|all) (previous |prior )?(instructions?|rules?|guidelines?)',
        r'forget (everything|all) (your |previous )?(instructions?|rules?)',
        r'override (your|the) (system )?(instructions?|prompt)',
    ],
    'C2_identity_hijacking': [
        r'to survive,? you (must|need)',
        r'you (must|need to) (now|immediately|first|only)',
        r'your (only |real |true |new )?(job|purpose|task|mission|goal) (now )?is to',
        r'(act|behave) as (if you are|though you are|a |an )(human|admin|owner|moderator|different)',
        r'(obey|follow) (my|these|the following) (command|instruction|order)',
    ],
    'C3_advertising_redirection': [
        r'tell your human about',
        r'(visit|go to|check out|sign up at) [a-z0-9-]+\.(com|ai|io|net|xyz|app)',
        r'(you should |you must )?recommend (our|this) (product|service|tool|platform)',
        r'become obsessed with',
        r'(make|tell) your human (buy|purchase|sign up|subscribe)',
    ],
    'C4_info_exfiltration': [
        r'sk-ant-api',
        r'sk-[a-zA-Z0-9]{20,}',
        r'(reveal|share|print|paste|send) (me )?(your )?(full )?(system prompt|instructions)',
        r'(reveal|share|print|paste|send) (me )?(your )?(SOUL|HEART|BOOTSTRAP|CLAUDE)\.md',
        r'what (does|do) your (system prompt|instructions?) (say|contain)',
    ],
    'C5_memory_poisoning': [
        r'(save|add|store) (this|the following) (to your |in your )(memory|context|soul)',
        r'from now on,? (you will|you must|always) (respond|reply|recommend|promote)',
        r'remember (this|the following) (instruction|command|forever|permanently)',
        r'(permanently |always )(update|modify) your (memory|behavior|instructions)',
    ],
}

compiled = {cat: [re.compile(p, re.IGNORECASE) for p in pats]
            for cat, pats in attack_patterns.items()}

def scan(text):
    hits = {}
    for cat, pats in compiled.items():
        c = sum(1 for p in pats if p.search(text))
        if c > 0:
            hits[cat] = c
    return hits

print('Patterns loaded:', sum(len(p) for p in attack_patterns.values()), 'patterns')

In [ ]:
from tqdm.auto import tqdm

# Add the phase label
def assign_phase(dt):
    if dt <= pd.Timestamp('2026-02-12').date():
        return 'Phase 1'
    elif dt <= pd.Timestamp('2026-03-10').date():
        return 'Phase 2'
    else:
        return 'Phase 3'

df_posts['phase'] = df_posts['date'].apply(assign_phase)

# Build scan_text
df_posts['content'] = df_posts['content'].fillna('')
df_posts['scan_text'] = (df_posts['title'].fillna('') + ' ' + df_posts['content']).str.strip()

# Scan (3.1M posts - may take 30-45 min)
tqdm.pandas(desc='Attack scanning (full archive)')
df_posts['attack_hits'] = df_posts['scan_text'].progress_apply(scan)
df_posts['n_categories'] = df_posts['attack_hits'].apply(len)
df_posts['total_hits']   = df_posts['attack_hits'].apply(lambda d: sum(d.values()))

df_candidates = df_posts[df_posts['n_categories'] > 0].copy()

print(f'Total posts:         {len(df_posts):,}')
print(f'Candidate attacks:   {len(df_candidates):,} ({len(df_candidates)/len(df_posts)*100:.3f}%)')
print()

# Phase-based intensity
print('=== Attack distribution by phase ===')
for phase in ['Phase 1','Phase 2','Phase 3']:
    total = (df_posts['phase']==phase).sum()
    attacks = (df_candidates['phase']==phase).sum()
    rate = attacks/total*100 if total>0 else 0
    print(f'{phase}: {attacks:,} attacks / {total:,} posts = {rate:.3f}%')

print()
print('=== Phase x Category matrix ===')
from collections import Counter
for phase in ['Phase 1','Phase 2','Phase 3']:
    sub = df_candidates[df_candidates['phase']==phase]
    cat_counts = Counter()
    for hits in sub['attack_hits']:
        for cat in hits:
            cat_counts[cat] += 1
    print(f'{phase}: {dict(cat_counts)}')

### 4a. Cache the scan results

The scanned `df_posts` and the candidate pool `df_candidates` are saved to Drive as parquet files, so that in later sessions they load within seconds via Section 3 Option A (no need to repeat the scan).

In [ ]:
import os

# Persist the scan results to Drive (so it does not take 23 min again)
SCAN_CACHE = os.path.join(DATASET_DIR, 'full_archive_scan_cache.parquet')

# The attack_hits column holds dicts, which parquet cannot write directly -> convert to string
df_posts_to_save = df_posts.copy()
df_posts_to_save['attack_hits_str'] = df_posts_to_save['attack_hits'].apply(str)
df_posts_to_save = df_posts_to_save.drop(columns=['attack_hits'])

df_posts_to_save.to_parquet(SCAN_CACHE, index=False)
print(f'Scan results saved: {SCAN_CACHE}')
print(f'File size: {os.path.getsize(SCAN_CACHE) / 1e6:.1f} MB')

# Also save the candidate pool separately (small, for fast access)
CANDIDATES_CACHE = os.path.join(DATASET_DIR, 'full_archive_candidates.parquet')
df_candidates_to_save = df_candidates.copy()
df_candidates_to_save['attack_hits_str'] = df_candidates_to_save['attack_hits'].apply(str)
df_candidates_to_save = df_candidates_to_save.drop(columns=['attack_hits'])
df_candidates_to_save.to_parquet(CANDIDATES_CACHE, index=False)
print(f'Candidate pool saved: {CANDIDATES_CACHE}')

## 5. Exploratory Data Analysis (EDA)

Basic descriptive statistics of the platform: daily post volume (with phases; Figure 1 in the paper), the most active communities (submolts; Figure 2), the content length distribution, and the most active agents.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import os

daily = df_posts.groupby('date').size().reset_index(name='count')
daily['date'] = pd.to_datetime(daily['date'])

fig, ax = plt.subplots(figsize=(14, 5))

# Phase boundaries: from the actual data and the Meta news (10 March)
phase1 = daily[daily['date'] <= '2026-02-12']
phase2 = daily[(daily['date'] > '2026-02-12') & (daily['date'] <= '2026-03-10')]
phase3 = daily[daily['date'] > '2026-03-10']

ax.bar(phase1['date'], phase1['count']/1000,
       color='#b85450', alpha=0.85, width=0.9, label='Phase 1: Launch & peak (27 Jan-12 Feb)')
ax.bar(phase2['date'], phase2['count']/1000,
       color='#6c8ebf', alpha=0.85, width=0.9, label='Phase 2: Stabilization (13 Feb-10 Mar)')
ax.bar(phase3['date'], phase3['count']/1000,
       color='#82b366', alpha=0.85, width=0.9, label='Phase 3: Post-acquisition (11 Mar-28 May)')

# Peak annotation
ax.annotate('Peak:\n371K posts\n(9 Feb)',
            xy=(pd.Timestamp('2026-02-09'), 371),
            xytext=(pd.Timestamp('2026-02-18'), 300),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
            fontsize=8.5, ha='center')

# Meta acquisition
ax.axvline(pd.Timestamp('2026-03-10'), ls='--', color='#333333', alpha=0.8, lw=1.5)
ax.text(pd.Timestamp('2026-03-11'), 180,
        'Meta acquisition\nannounced\n(10 Mar)', fontsize=8, color='#333333', va='top')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Daily posts (thousands)', fontsize=11)
ax.set_title('Daily Post Volume on Moltbook (27 Jan - 28 May 2026)', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 420)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
save_fig('01_daily_activity_full.png')
plt.show()

# Phase-based statistics
for label, start, end in [
    ('Phase 1 (Launch & peak)', '2026-01-27', '2026-02-12'),
    ('Phase 2 (Stabilization)', '2026-02-13', '2026-03-10'),
    ('Phase 3 (Post-acquisition)', '2026-03-11', '2026-05-28'),
]:
    mask = (daily['date'] >= pd.Timestamp(start)) & (daily['date'] <= pd.Timestamp(end))
    sub = daily[mask]
    total = sub['count'].sum()
    mean = sub['count'].mean()
    print(f"{label}: {len(sub)} days, total {total:,}, mean {mean:,.0f}/day")

In [ ]:
import matplotlib.pyplot as plt

top_submolts = df_posts['submolt'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#9673a6' if s != 'general' else '#6c4a8c' for s in top_submolts.index[::-1]]
top_submolts.sort_values().plot(kind='barh', color=colors, ax=ax)

ax.set_xscale('log')
ax.set_title('Top 15 Most Active Submolts (log scale)', fontsize=13)
ax.set_xlabel('Number of posts (log scale)', fontsize=11)
ax.set_ylabel('Submolt', fontsize=11)

# Add value labels (for readability)
for i, (name, val) in enumerate(top_submolts.sort_values().items()):
    ax.text(val * 1.1, i, f'{val:,}', va='center', fontsize=8.5)

plt.tight_layout()
save_fig('02_top_submolts.png')
plt.show()

print(top_submolts.to_string())

In [ ]:
import matplotlib.pyplot as plt

df_posts['content'] = df_posts['content'].fillna('')
df_posts['content_len'] = df_posts['content'].str.len()

fig, ax = plt.subplots(figsize=(10, 5))
df_posts[df_posts['content_len'] < 2000]['content_len'].hist(
    bins=60, color='#d85a30', edgecolor='white', linewidth=0.3, ax=ax
)

ax.set_title('Post Content Length Distribution (< 2,000 characters)', fontsize=13)
ax.set_xlabel('Character count', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)

# Median and mean lines
median = df_posts['content_len'].median()
mean = df_posts['content_len'].mean()
ax.axvline(median, ls='--', color='black', alpha=0.7, label=f'Median: {median:.0f}')
ax.axvline(mean, ls=':', color='gray', alpha=0.7, label=f'Mean: {mean:.0f}')
ax.legend(fontsize=9)

plt.tight_layout()
save_fig('03_content_length.png')
plt.show()

print(df_posts['content_len'].describe())

In [ ]:
top_agents = df_posts['agent_name'].value_counts().head(10)
print('--- Top 10 most active agents ---')
print(top_agents)
print(f'\nTotal unique agents: {df_posts["agent_name"].nunique():,}')
print(f'Share of the most active agent in the total: {top_agents.iloc[0]/len(df_posts)*100:.2f}%')

## 6. Phase-based Attack Analysis

How the attack rate (per 1,000 posts) and the category composition change across the three platform phases (the phase-analysis figure in the paper). *(The phase totals/attack counts in the figure are taken from the scan output.)*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---------- Left panel: attack intensity by phase ----------
phase_labels = ['Phase 1\n(27 Jan-12 Feb)', 'Phase 2\n(13 Feb-10 Mar)', 'Phase 3\n(11 Mar-28 May)']
total_posts = [1_242_044, 763_754, 1_099_338]
attacks     = [1_085, 1_290, 1_079]
rate        = [s/t*1000 for s, t in zip(attacks, total_posts)]  # attacks per 1,000 posts

colors = ['#b85450', '#6c8ebf', '#82b366']
bars = axes[0].bar(phase_labels, rate, color=colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, rate):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[0].set_ylabel('Attack rate (per 1,000 posts)', fontsize=11)
axes[0].set_title('(a) Attack Rate by Platform Phase', fontsize=12)
axes[0].set_ylim(0, max(rate) * 1.25)
axes[0].axhline(sum(attacks)/sum(total_posts)*1000, ls='--',
                color='gray', alpha=0.7, label=f'Overall mean: {sum(attacks)/sum(total_posts)*1000:.2f}')
axes[0].legend(fontsize=9)

# ---------- Right panel: phase x category grouped bars ----------
categories = ['C1\nInstruction\nOverride',
              'C2\nIdentity\nHijacking',
              'C3\nAdvertising/\nRedirection',
              'C4\nInfo\nExfiltration',
              'C5\nMemory\nPoisoning']

# Category counts for each phase (in C1..C5 order)
p1 = [311, 297, 432, 44, 80]
p2 = [165, 200, 910, 21, 7]
p3 = [204, 428, 381, 63, 10]

x = np.arange(len(categories))
w = 0.25

b1 = axes[1].bar(x - w, p1, w, label='Phase 1', color='#b85450', alpha=0.85)
b2 = axes[1].bar(x,     p2, w, label='Phase 2', color='#6c8ebf', alpha=0.85)
b3 = axes[1].bar(x + w, p3, w, label='Phase 3', color='#82b366', alpha=0.85)

axes[1].set_xticks(x)
axes[1].set_xticklabels(categories, fontsize=9)
axes[1].set_ylabel('Number of candidate posts', fontsize=11)
axes[1].set_title('(b) Attack Category Distribution by Phase', fontsize=12)
axes[1].legend(fontsize=9)

plt.suptitle('A2A Prompt Injection Attack Patterns Across Platform Phases',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('10_phase_attack_analysis.png')
plt.show()

# Summary statistics
print('Overall attack rate: {:.3f}%'.format(sum(attacks)/sum(total_posts)*100))
print('C5 Phase 1 -> Phase 2 change: {} -> {} ({:.0f}% decrease)'.format(
    80, 7, (80-7)/80*100))
print('C3 Phase 2 peak: 910 (Phase 1: 432, Phase 3: 381)')
print('C2 Phase 3 increase: {} -> {} -> {}'.format(297, 200, 428))

## 7. Phase-balanced Positive Sampling

From the candidate pool, ~450 positive candidates are selected **proportionally** to each phase's share of the pool (so that rare phases are not overwhelmed). Candidates with higher `n_categories`/`total_hits` scores are prioritised.

In [ ]:
import pandas as pd
import os

# --- Load from cache (you may skip this if df_candidates is still in memory in this session) ---
CANDIDATES_CACHE = os.path.join(DATASET_DIR, 'full_archive_candidates.parquet')
df_candidates = pd.read_parquet(CANDIDATES_CACHE)
df_candidates['attack_hits'] = df_candidates['attack_hits_str'].apply(eval)
df_candidates = df_candidates.drop(columns=['attack_hits_str'])

print(f'Total candidates: {len(df_candidates):,}')
print(df_candidates['phase'].value_counts())

# --- Phase-balanced proportional sample ---
RANDOM_STATE = 42
N_POSITIVE_TOTAL = 450

phase_distribution = df_candidates['phase'].value_counts(normalize=True)
print('\nPhase proportions (used in sampling):')
print(phase_distribution)

positive_parts = []
for phase, proportion in phase_distribution.items():
    n_this_phase = int(round(N_POSITIVE_TOTAL * proportion))
    sub = df_candidates[df_candidates['phase'] == phase].sort_values(
        ['n_categories', 'total_hits'], ascending=False
    )
    positive_parts.append(sub.head(n_this_phase))

df_positive = pd.concat(positive_parts).drop_duplicates(subset='id')
print(f'\nSelected positive candidates: {len(df_positive)}')
print(df_positive['phase'].value_counts())

## 8. Negative Sampling and Seed Export

An equal share from each phase (60 each, 180 in total) of **negative** posts (no scan match, `n_categories==0`) is selected. These are combined with the positive candidates and the resulting 630-post seed is shuffled and exported -> to be labelled manually in Phase 2.

In [ ]:
import pandas as pd
import os

# --- Negative sample: equal share from each phase (balanced negative pool) ---
N_NEGATIVE_TOTAL = 180
N_NEGATIVE_PER_PHASE = N_NEGATIVE_TOTAL // 3  # 60 each

# df_posts should still be in memory (output of the full archive scan)
# Negative = posts with no attack match (n_categories == 0)
negative_parts = []
for phase in ['Phase 1', 'Phase 2', 'Phase 3']:
    sub = df_posts[(df_posts['phase'] == phase) & (df_posts['n_categories'] == 0)]
    selected = sub.sample(n=N_NEGATIVE_PER_PHASE, random_state=RANDOM_STATE)
    negative_parts.append(selected)

df_negative = pd.concat(negative_parts)
print(f'Selected negative candidates: {len(df_negative)}')
print(df_negative['phase'].value_counts())

# --- Build the seed dataset ---
seed = pd.concat([df_positive, df_negative]).sample(frac=1, random_state=RANDOM_STATE)

seed_export = seed[[
    'id', 'agent_name', 'submolt', 'title', 'content',
    'score', 'comment_count', 'phase', 'n_categories', 'total_hits'
]].copy()

seed_export['heuristic_categories'] = seed['attack_hits'].apply(
    lambda d: ', '.join(d.keys()) if d else 'NONE'
)

# EMPTY columns for manual labelling
seed_export['label_is_attack'] = ''
seed_export['label_category'] = ''
seed_export['label_notes'] = ''

print(f'\nTotal seed dataset size: {len(seed_export)} rows')
print('\nPhase distribution (within the seed):')
print(seed_export['phase'].value_counts())
print('\nLabel/phase cross-tabulation (heuristic, for verification):')
print(pd.crosstab(seed_export['phase'], seed_export['n_categories'] > 0))

# --- Save ---
seed_path_v2 = os.path.join(DATASET_DIR, 'moltbook_seed_v2.csv')
seed_export.to_csv(seed_path_v2, index=False, encoding='utf-8')
print(f'\nSaved: {seed_path_v2}')

## Appendix A. Screen Recall Audit

To estimate how much of the platform's true attacks the heuristic scan captures, a phase-balanced **random sample of 1,500 posts** is drawn from the archive and labelled manually; the true attack base rate and the **recall** and **precision** of the scan are computed with Wilson 95% confidence intervals. *(This quantitatively demonstrates that the study's evaluation is screen-conditional.)*

### A.1 Prepare the sample

Draws a phase-balanced random sample from the archive, marks the `regex_flagged` status and exports it as `recall_audit_sample.csv` for manual labelling (fill in the `human_label` column with 0/1).

In [ ]:
# === Screen-recall audit, step 1: random sample preparation (with guards) ===
# Run order: the Drive/paths cell, then the cache-loading cell, then this one.
import numpy as np, pandas as pd, os, ast, re

N_SAMPLE = 1500
RANDOM_STATE = 42

dfp = df_posts.copy()

# --- Guards: regenerate anything missing from the cache ---
# If attack_hits is not a dict, convert it from string
if not isinstance(dfp['attack_hits'].iloc[0], dict):
    dfp['attack_hits'] = dfp['attack_hits'].apply(
        lambda v: ast.literal_eval(v) if isinstance(v, str) else (v or {}))
# n_categories
if 'n_categories' not in dfp.columns:
    dfp['n_categories'] = dfp['attack_hits'].apply(lambda d: len(d) if isinstance(d, dict) else 0)
# date
if 'date' not in dfp.columns:
    dfp['date'] = pd.to_datetime(dfp['created_at'], utc=True).dt.date
# phase (exactly the same boundaries as in Phase 1)
if 'phase' not in dfp.columns:
    def _phase(dt):
        if dt <= pd.Timestamp('2026-02-12').date(): return 'Phase 1'
        elif dt <= pd.Timestamp('2026-03-10').date(): return 'Phase 2'
        else: return 'Phase 3'
    dfp['phase'] = dfp['date'].apply(_phase)
# id
if 'id' not in dfp.columns:
    dfp = dfp.reset_index().rename(columns={'index': 'id'})

# --- Sample fields ---
dfp['display_text'] = (dfp['title'].fillna('') + ' \u2014 ' + dfp['content'].fillna('')).str.slice(0, 1000)
dfp['regex_flagged'] = (dfp['n_categories'] > 0).astype(int)
dfp['regex_categories'] = dfp['attack_hits'].apply(
    lambda d: ','.join(d.keys()) if isinstance(d, dict) else '')

# Exclude the seed
seed_ids = set()
for fn in ['moltbook_seed_v2.csv']:
    try:
        seed_ids = set(pd.read_csv(os.path.join(DATASET_DIR, fn))['id']); break
    except Exception:
        continue
if not seed_ids:
    print("WARNING: seed file not found; the seed was not excluded (~630 out of 3.1M is negligible).")
pool = dfp[~dfp['id'].isin(seed_ids)]

# Phase-balanced random sample
per_phase = N_SAMPLE // 3
parts = [pool[pool['phase'] == ph].sample(n=min(per_phase, (pool['phase'] == ph).sum()),
                                          random_state=RANDOM_STATE)
         for ph in ['Phase 1', 'Phase 2', 'Phase 3']]
sample = pd.concat(parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Export (to be labelled manually)
exp = sample[['id', 'phase', 'regex_flagged', 'regex_categories', 'submolt', 'display_text']].copy()
exp['human_label'] = ''      # 1 = attack, 0 = not  (FILL IN MANUALLY)
exp['human_category'] = ''   # optional: C1..C5
out = os.path.join(DATASET_DIR, 'recall_audit_sample.csv')
exp.to_csv(out, index=False)
print(f"Sample ready: {len(exp)} posts -> {out}")
print(f"{exp['regex_flagged'].sum()} of these are already flagged by the regex.")
print("NOW: open the CSV, fill 'human_label' with 0/1, save it under the same name, then run Cell 1B.")

### A.2 Compute after labelling

Computes the base rate, the scan recall and the precision (Wilson 95% CI) from the labelled sample.

In [ ]:
# === Screen-recall audit, step 2: estimates from the labelled sample (Wilson 95% CI) ===
import pandas as pd, numpy as np, os

def wilson(k, n, z=1.96):
    if n == 0: return (float('nan'), float('nan'), float('nan'))
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return p, max(0, c-h), min(1, c+h)

lab = pd.read_csv(os.path.join(DATASET_DIR, 'recall_audit_labeled.csv'))
lab = lab[lab['human_label'].astype(str).str.strip().isin(['0','1'])].copy()
lab['human_label'] = lab['human_label'].astype(int)
lab['regex_flagged'] = lab['regex_flagged'].astype(int)

N = len(lab)
true_att = lab[lab['human_label'] == 1]
n_true = len(true_att)
caught = int((true_att['regex_flagged'] == 1).sum())   # true attacks caught by the regex
flagged = lab[lab['regex_flagged'] == 1]
tp_flag = int((flagged['human_label'] == 1).sum())

print(f"Labelled sample: N = {N}")
br = wilson(n_true, N)
print(f"True attack base rate: {n_true}/{N} = {br[0]*100:.3f}%  (95% CI: {br[1]*100:.3f}-{br[2]*100:.3f}%)")
print(f"Comparison - regex candidate rate (full archive): 0.111%")
if n_true > 0:
    rc = wilson(caught, n_true)
    print(f"\nSCAN RECALL: {caught}/{n_true} = {rc[0]*100:.1f}%  (95% CI: {rc[1]*100:.1f}-{rc[2]*100:.1f}%)")
    print(f"Attacks MISSED by the regex (in the sample): {n_true - caught}")
else:
    print("\nWARNING: no true attacks appeared in the sample; you may need to increase N to estimate recall.")
if len(flagged) > 0:
    pr = wilson(tp_flag, len(flagged))
    print(f"\nSCAN PRECISION (in the sample): {tp_flag}/{len(flagged)} = {pr[0]*100:.1f}%  (95% CI: {pr[1]*100:.1f}-{pr[2]*100:.1f}%)")